# Conformal geometry

By default, Wakis assigns one material per mesh cell (**staircase** geometry). A curved surface is then replaced by the faces of the cells it passes through, which changes the geometry by up to one cell size. For curved conductors and dielectrics this shifts resonance frequencies and limits how fast results converge with mesh refinement.

With `geometry_mode="conformal"` Wakis instead intersects the STL surfaces with the grid (conformal FIT):

* **PEC walls cut the mesh**: an edge cut by a PEC wall keeps only its length inside the vacuum or dielectric, and a cut face keeps only its area there (a Dey-Mittra-type cut-cell scheme).
* **Dielectric and magnetic interfaces** use effective materials: a mesh edge crossing an interface gets a mix of the two materials, weighted by how the interface cuts it.

This notebook shows the new options on the cavity of notebook 001, a vacuum cavity inside a lossy shell:

| | |
|---|---|
| `geometry_mode` | `"staircase"` (default) or `"conformal"` |
| `stl_material_types` | `"normal"`, `"pec"` (and `"sibc"`) per STL solid |
| `bg_material_type` | type of the background material |
| `conformal_stabilization` | `"area_floor"` keeps the time step stable with small cut cells |
| `solver.compute_stable_timestep()` | exact stability limit of the time step |
| `solver.plot_mesh_materials()` | the mesh, its materials and the STL geometry in one plane |

In [1]:
import tempfile
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv

from wakis import GridFIT3D, SolverFIT3D, WakeSolver

# Show warnings as 'UserWarning: message', without the source file location.
warnings.formatwarning = lambda message, category, *args, **kwargs: (
    f"{category.__name__}: {message}\n"
)

## Geometry

We read the two STL files of notebook 001: the vacuum cavity, a pillbox with rounded edges and beam pipes, and the shell around it, made of a lossy material ($\varepsilon_r = 10$, $\sigma = 10$ S/m). Outside the shell is the PEC background. As in notebook 001, the simulation domain is the bounding box of the two solids, so the beam pipes end at the domain boundary.

In [2]:
stl_cavity = "data/001_vacuum_cavity.stl"
stl_shell = "data/001_lossymetal_shell.stl"
stl_solids = {"cavity": stl_cavity, "shell": stl_shell}
stl_materials = {"cavity": "vacuum", "shell": [10, 1.0, 10]}

surf = pv.read(stl_shell) + pv.read(stl_cavity)
xmin, xmax, ymin, ymax, zmin, zmax = surf.bounds

## Grid and solver setup

The conformal grid needs to know how each solid takes part in the cut-cell treatment:

* `stl_material_types`: `"normal"` for vacuum, dielectric or magnetic solids (they are kept and get effective materials at their interfaces), `"pec"` for perfect conductors (they cut the mesh). Here both solids are `"normal"`.
* `bg_material_type="pec"`: the background around the solids is a perfect conductor, so the outer surface of the shell is a PEC wall cutting the mesh. It must match the solver background `bg="pec"`.
* `conformal_stabilization="area_floor"`: explained in the stability section below.

The staircase grid uses the same solids and materials. Both classify the grid nodes with `stl_method="voxelize_rectilinear"`, which needs a mesh with uniform spacing along each axis. The mesh is coarser than in notebook 001 to keep this notebook fast.

In [3]:
mesh = dict(
    xmin=xmin, xmax=xmax, ymin=ymin, ymax=ymax, zmin=zmin, zmax=zmax,
    Nx=50, Ny=50, Nz=80,
    stl_method="voxelize_rectilinear", verbose=0,
)  # fmt: skip

# Conformal geometry options (see above)
conformal = dict(
    geometry_mode="conformal",
    stl_material_types={"cavity": "normal", "shell": "normal"},
    bg_material_type="pec",
    conformal_stabilization="area_floor",
)

# Closed cavity: PEC background and PEC domain boundaries
cavity = dict(
    bg="pec", bc_low=["pec"] * 3, bc_high=["pec"] * 3,
    use_stl=True, dtype=np.float64, verbose=0,
)  # fmt: skip

grids = {
    "staircase": GridFIT3D(**mesh, stl_solids=stl_solids, stl_materials=stl_materials),
    "conformal": GridFIT3D(
        **mesh, stl_solids=stl_solids, stl_materials=stl_materials, **conformal
    ),
}
solvers = {mode: SolverFIT3D(grid, **cavity) for mode, grid in grids.items()}

Generating grid with 200000 mesh cells...
Generating grid with 200000 mesh cells...


17 edges contain material layers thinner than the mesh resolution; at most one interface per edge is represented. Refine the mesh there if these layers matter.


Initializing Electromagnetic solver...
Initializing Electromagnetic solver...


The shell STL contains a tiny crack (about 0.1 mm), so Wakis warns that it is not a closed surface. The crack is far below the cell size and does not change the mesh. The conformal grid also reports a few edges where the shell is thinner than a mesh cell; the geometry diagnostics section explains this warning.

## Inspecting the geometry on the mesh

`solver.plot_mesh_materials()` shows one grid plane with the mesh, the material of every face and the exact STL cross-section (black lines). In FIT, the material of an electric or magnetic DOF belongs to the face its edge pierces, so there are two views:

* `grid_type="primal"`: primal faces, coloured by $\mu_r$. Faces cut by a PEC wall are coloured only in their part inside the cavity (the PEC part is grey), and the mesh edges stop at the wall.
* `grid_type="dual"`: dual faces, coloured by $\varepsilon_r$ (or $\sigma$ with `quantity="sigma"`). Dual faces are never cut; faces whose electric edge lies in PEC are grey.

The plane normal selects the tensor component: in an $x$-normal plane the faces carry $\varepsilon_{r,x}$.

**PEC wall.** The primal $z$-plane through the middle of the cavity, with the shell between the two circles: the staircase follows the cell faces, the conformal mesh follows the outer circle, the PEC wall.

In [4]:
k_middle = int(np.argmin(np.abs(grids["conformal"].z - 0.15)))
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, (mode, solver) in zip(axes, solvers.items()):
    solver.plot_mesh_materials(grid_type="primal", normal="z", index=k_middle, ax=ax)
    ax.set_title(f"{mode}: " + ax.get_title())
fig.tight_layout()

**Material interface.** The dual $x$-plane through the axis shows $\varepsilon_{r,x}$ along the whole cavity. The staircase switches between 1 (cavity) and 10 (shell) from one cell to the next; the conformal mesh gives the edges crossing the inner surface of the shell an effective value in between. It depends on where the surface cuts the edge and on its orientation: the field normal to the surface sees the harmonic mean of the two materials (series), the field along it the arithmetic mean (parallel).

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(15, 8))
for ax, (mode, solver) in zip(axes, solvers.items()):
    solver.plot_mesh_materials(
        grid_type="dual", normal="x", index=mesh["Nx"] // 2, quantity="eps", ax=ax
    )
    ax.set_title(f"{mode}: " + ax.get_title())
fig.tight_layout()

Red outlines mark the conformal stabilization (next section). In Jupyter, `interactive=True` adds buttons for the plane normal (and the material) and a slider to move through the planes; it needs an interactive backend such as `%matplotlib widget`:

```python
solvers["conformal"].plot_mesh_materials(grid_type="dual", interactive=True)
```

## Geometry diagnostics

While building a conformal grid, Wakis checks every mesh edge against the STL surfaces and warns about two kinds of edges:

* **Thin feature**: the edge contains material structure thinner than the mesh, e.g. a layer that lies completely between the two edge ends. At most one interface per edge is represented, so the layer is partly or completely lost.
* **Inconsistent**: the STL crossings found on the edge do not match the inside/outside classification of its end points, e.g. at an open (non-watertight) STL surface. These edges keep the staircase material or an ambiguous cut.

The warning only gives the number of edges. `plot_mesh_materials()` shows where they are: orange circles mark thin features, magenta crosses inconsistent edges, and the legend counts them in the plane. We add two flawed geometries to the cavity, one for each kind.

**Thin feature: a thin dielectric window.** A dielectric disc ($\varepsilon_r = 3$) across the downstream beam pipe, 0.3 cells thick and placed between two node planes, so no grid node lies inside it.

In [6]:
stl_folder = tempfile.mkdtemp()
z_nodes = grids["conformal"].z
dz = z_nodes[1] - z_nodes[0]
k_window = int(np.argmin(np.abs(z_nodes - 0.45)))
stl_solids_window = {**stl_solids, "window": f"{stl_folder}/window.stl"}
pv.Cylinder(
    center=(0, 0, z_nodes[k_window] + 0.25 * dz),
    radius=0.06,  # beam pipe radius
    height=0.3 * dz,
    direction=(0, 0, 1),
    resolution=100,
).triangulate().save(stl_solids_window["window"])

grid_window = GridFIT3D(
    **mesh,
    stl_solids=stl_solids_window,
    stl_materials={**stl_materials, "window": [3.0, 1.0, 0.0]},
    **conformal,
)
solver_window = SolverFIT3D(grid_window, **cavity)

Generating grid with 200000 mesh cells...


113 edges contain material layers thinner than the mesh resolution; at most one interface per edge is represented. Refine the mesh there if these layers matter.


Initializing Electromagnetic solver...


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
solver_window.plot_mesh_materials(grid_type="primal", normal="y", ax=axes[0])
solver_window.plot_mesh_materials(
    grid_type="dual", normal="z", index=k_window + 1, ax=axes[1]
)
fig.tight_layout()

The side view (left, primal $y$-plane) marks the $z$-edges through the window. The dual $z$-plane through the window (right) shows the consequence: the faces pierced by these edges still have $\varepsilon_{r,z} = 1$, so the window is missing from the simulation. Refining the mesh until the window spans at least about one cell fixes this.

**Inconsistent: an STL with a hole.** The shell is exported with a hole in its outer wall, as can happen with a damaged CAD export. Wakis warns that the surface is not closed and that edges are inconsistent:

In [8]:
shell = pv.read(stl_shell)
centers = shell.cell_centers().points
angle = np.degrees(np.arctan2(centers[:, 1], centers[:, 0]))
hole = (
    (np.abs(np.hypot(centers[:, 0], centers[:, 1]) - 0.26) < 3e-3)  # outer wall
    & (np.abs(angle - 30) < 12)
    & (np.abs(centers[:, 2] - 0.15) < 0.03)
)
stl_solids_hole = {**stl_solids, "shell": f"{stl_folder}/shell_hole.stl"}
shell.extract_cells(np.nonzero(~hole)[0]).extract_surface(
    algorithm="dataset_surface"
).save(stl_solids_hole["shell"])

grid_hole = GridFIT3D(
    **mesh, stl_solids=stl_solids_hole, stl_materials=stl_materials, **conformal
)
solver_hole = SolverFIT3D(grid_hole, **cavity)

Generating grid with 200000 mesh cells...


17 edges contain material layers thinner than the mesh resolution; at most one interface per edge is represented. Refine the mesh there if these layers matter.
155 edges have STL crossings that disagree with the grid point classification and receive no or an ambiguous conformal correction. Check that the STL solids are closed and not tangent to grid points.


Initializing Electromagnetic solver...


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
solver_hole.plot_mesh_materials(
    grid_type="primal", normal="z", index=k_middle, ax=axes[0]
)
solver_hole.plot_mesh_materials(
    grid_type="dual", normal="z", index=k_middle, ax=axes[1]
)
fig.tight_layout()

The STL cross-section (black) shows the gap in the outer wall of the shell. The voxelization still classifies all grid nodes as for the closed shell, but grid lines passing through the gap find no surface to cut: their edges change material without a crossing. The crosses mark these edges. They get no conformal cut and take their material from their end points, so the geometry falls back to a staircase at the hole, while it stays conformal elsewhere. How an open surface is classified depends on the shape of the hole, so repair the STL (make it watertight) before simulating.

## Stability of the time step

The explicit leapfrog update is stable only for $\Delta t < 2/\omega_{\max}$, where $\omega_{\max}$ is the highest eigenfrequency of the discrete system. On an uncut grid this is the CFL limit. A face cut by a PEC wall into a tiny piece makes $\omega_{\max}$ very large, and the time step needed for stability would become tiny.

`conformal_stabilization="area_floor"` raises every cut face to at least a fraction (default 0.1) of its full area, and removes edges with less than that fraction left in the cavity. This bounds the time step reduction at the cost of a small, local geometry change. `compute_stable_timestep()` computes the exact stability limit of the assembled solver and compares it with the time step in use:

In [10]:
cases = {
    "staircase": solvers["staircase"],
    "conformal, no stabilization": SolverFIT3D(
        GridFIT3D(
            **mesh,
            stl_solids=stl_solids,
            stl_materials=stl_materials,
            **{**conformal, "conformal_stabilization": None},
        ),
        **cavity,
    ),
    "conformal, area floor": solvers["conformal"],
}
for name, solver in cases.items():
    result = solver.compute_stable_timestep(verbose=False)
    print(
        f"{name:28s} dt_max / dt_CFL = {result['dt_max'] / result['dt_cfl']:.3f}   "
        f"dt / dt_max = {result['dt_ratio']:.2f}   stable: {result['stable']}"
    )
print(
    f"area floor: {solvers['conformal'].n_area_floored} faces raised, "
    f"{solvers['conformal'].n_edges_eliminated} edges removed"
)

Generating grid with 200000 mesh cells...


Initializing Electromagnetic solver...


staircase                    dt_max / dt_CFL = 1.001   dt / dt_max = 0.44   stable: True


conformal, no stabilization  dt_max / dt_CFL = 0.120   dt / dt_max = 3.68   stable: False


conformal, area floor        dt_max / dt_CFL = 1.001   dt / dt_max = 0.44   stable: True
area floor: 4117 faces raised, 930 edges removed


Without stabilization, the cut faces of this cavity lower the stable time step to 12 % of the CFL limit. The solver's default time step (half the CFL limit, `cfln=0.5`) is then about four times too large, and the simulation would diverge. With the area floor, the stable time step stays at the CFL limit of the uncut grid, at the cost of a small, local geometry change.

## Wakefield simulation

We run the same wakefield simulation on both grids as in notebook 001: a Gaussian bunch ($\sigma_z = 10$ cm) on axis, here with 10 m of wake. The lossy shell damps the cavity modes, so the impedance shows broad peaks. The simulations print progress bars; `%%capture` keeps the notebook short.

In [11]:
%%capture
impedance = {}
for mode, grid in grids.items():
    folder = tempfile.mkdtemp()
    wake = WakeSolver(
        q=1e-9,
        sigmaz=0.1,
        beta=1.0,
        save=False,
        results_folder=f"{folder}/",
        Ez_file=f"{folder}/Ez.h5",
        verbose=0,
    )
    solver = SolverFIT3D(grid, wake, **cavity)
    solver.wakesolve(wakelength=10.0, plot=False)
    wake.calc_long_Z(samples=5001, fmax=1.2e9)
    impedance[mode] = (np.asarray(wake.f), np.abs(wake.Z))

In [12]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for mode, (f, Z) in impedance.items():
    ax.plot(f / 1e9, Z, label=mode)
ax.set(xlabel="f [GHz]", ylabel=r"$|Z_\parallel|$ [$\Omega$]")
ax.legend()

# Frequency of the strongest peak in each window
for f_low, f_high in [(0.4e9, 0.65e9), (0.65e9, 0.95e9), (0.95e9, 1.2e9)]:
    peaks = []
    for mode, (f, Z) in impedance.items():
        window = (f > f_low) & (f < f_high)
        peaks.append(f"{mode} {f[window][np.argmax(Z[window])] / 1e6:.0f} MHz")
    print(", ".join(peaks))

staircase 529 MHz, conformal 529 MHz
staircase 781 MHz, conformal 782 MHz
staircase 1089 MHz, conformal 1093 MHz


Both grids give the same mode frequencies to within about 0.4 %. The peak heights differ: they are set by the losses in the shell, and the conformal mesh represents the shell surface with effective materials instead of whole cells. This notebook has no reference to tell which result is closer. The accuracy of the conformal scheme is checked against analytic cavity modes in the Wakis tests (`tests/test_020_conformal.py`): on the same coarse mesh the conformal frequency error of a pillbox cavity is about sixteen times smaller than the staircase error, and it decreases about with the square of the cell size.

## Summary

* `geometry_mode="conformal"` represents curved PEC walls and material interfaces inside the mesh cells, with the same STL files and grid as the staircase geometry.
* Each STL solid needs a `stl_material_types` entry: `"normal"` solids get effective materials at their interfaces, `"pec"` solids cut the mesh. `bg_material_type` sets the type of the background and must match the solver background `bg`.
* Keep `conformal_stabilization="area_floor"` for time-domain simulations with PEC cut cells; `compute_stable_timestep()` checks the resulting time step. Without it, the cut cells of this cavity make the default time step unstable.
* `plot_mesh_materials()` is the quickest way to check how the mesh represents a geometry before running a simulation.
* Check the geometry diagnostics warnings: `plot_mesh_materials()` marks where layers thinner than the mesh are lost and where the STL is not consistent (e.g. not watertight).

Current limitations: each mesh edge is cut at most once (an edge crossing two surfaces is reported in a warning and treated approximately), and the effective materials at dielectric interfaces are diagonal approximations of the full tensor.